In [11]:
import pandas as pd
import os
import re
import tarfile
from transformers import BigBirdTokenizer
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
# # NOTE: Run this during setup the first time.
# tar_path = './data/UNGDC_1946-2023.tgz'
# extract_path = './data/UNGDC_1946-2023/'

# with tarfile.open(tar_path, 'r:gz') as tar:
#     tar.extractall(path=extract_path)

In [3]:
vdem_df = pd.read_csv('./data/V-Dem-CY-Full+Others-v14.csv')

C:\Users\maxla\AppData\Local\Temp\ipykernel_64384\952016332.py:1: DtypeWarning: Columns (364,365,366,399,415,804,836,837,924,1240,1257,1486,3094,3168,3169,3341,3342,3344,3345,3347,3350,3352) have mixed types. Specify dtype option on import or set low_memory=False.
  vdem_df = pd.read_csv('./data/V-Dem-CY-Full+Others-v14.csv')


In [4]:
base_dir = './data/UNGDC_1946-2023/TXT'

speeches = []

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith('.txt'):
            # The expected naming pattern is: COUNTRY_SESSION_YEAR.txt
            # e.g., USA_75_2020.txt
            match = re.match(r"([A-Z]{3})_(\d{1,3})_(\d{4})\.txt", file)
            if match:
                country, session, year = match.groups()
            else:
                continue

            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    text = f.read()
            except UnicodeDecodeError as e:
                print(f"Decoding error in {file_path}: {e}")
                continue

            speeches.append({
                'Country': country,
                'Session': int(session),
                'Year': int(year),
                'Text': text,
            })

df_speeches = pd.DataFrame(speeches)
print(df_speeches.head())

  Country  Session  Year                                               Text
0     ARG        1  1946  At the resumption of the first session of the ...
1     AUS        1  1946  The General Assembly of the United Nations is ...
2     BEL        1  1946  The\tprincipal organs of the United Nations ha...
3     BLR        1  1946  As more than a year has elapsed since the Unit...
4     BOL        1  1946  Coming to this platform where so many distingu...


In [5]:
df_speeches['Year'] = df_speeches['Year'].astype(int)
vdem_df['year'] = vdem_df['year'].astype(int)

In [6]:
df = pd.merge(
    df_speeches,
    vdem_df,
    left_on=['Country', 'Year'],
    right_on=['country_text_id', 'year'],
    how='left'
)

# Please note that there are some NaN in the year etc (about 40 for year). See if is issue?

In [7]:
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df = df.dropna(subset=["v2x_polyarchy"])
df = df.sort_values(["Country", "Year"])

### Make Dataset

In [8]:
# Using the RoW metric they used in V-Forcast

df = df[['Country', 'Session', 'Year', 'Text', 'country_name', 'v2x_regime']]
df = df.sort_values(['Country', 'Year']).reset_index(drop=True)

def label_decrease_in_next_two_years(group):
    """
    For each row in this country's subgroup, check if v2x_regime
    is strictly lower in either the next 1 year or 2 years.
    """
    x = group['v2x_regime']
    decreased_next_2 = ((x.shift(-1) < x) | (x.shift(-2) < x)).astype(int)
    return decreased_next_2
    
df['Y'] = df.groupby('Country', group_keys=False).apply(label_decrease_in_next_two_years)

C:\Users\maxla\AppData\Local\Temp\ipykernel_64384\876506859.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['Y'] = df.groupby('Country', group_keys=False).apply(label_decrease_in_next_two_years)


In [9]:
model_name = "google/bigbird-roberta-base"
tokenizer = BigBirdTokenizer.from_pretrained(model_name)

encodings = tokenizer(
    df["Text"].tolist(),
    padding=True,       
    truncation=True,       # Truncate if over max length
    max_length=4096, 
    return_tensors="pt" 
)

c:\Users\maxla\OneDrive\Documents\GitHub\RCM\Speech-Gov-Prediction\venv\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\maxla\.cache\huggingface\hub\models--google--bigbird-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [12]:
labels = torch.tensor(df["Y"].values, dtype=torch.long)

class BigBirdTextDataset(Dataset):
    def __init__(self, encodings, labels, df_metadata):
        self.encodings = encodings
        self.labels = labels
        self.metadata = df_metadata.reset_index(drop=True)
    
    def __getitem__(self, idx):
        item = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx],
            # Add any metadata you want
            'country': self.metadata.loc[idx, 'Country'],
            'year': self.metadata.loc[idx, 'Year'],
        }
        return item

    def __len__(self):
        return len(self.labels)

dataset = BigBirdTextDataset(encodings, labels, df)

In [ ]:
torch.save(dataset, "data/dataset.pt")